# BERT-based Sarcasm Detection

Этот блокнот основан на разработанной ранее базовой модели (TF-IDF + логистическая регрессия).

Цель:
- Уловить контекстное значение текста
- Улучшить эффективность обнаружения сарказма
- Сравнить результаты с базовой моделью

In [3]:
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset


from sklearn.model_selection import train_test_split

## Загрузка данных

Для ускорения экспериментов используется заранее подготовленный сэмпл датасета (~30k–100k строк).

Сэмпл был получен из исходного датасета Reddit Sarcasm.

In [4]:
df = pd.read_csv("./df_sample.csv")
df.head()

,label,comment,author,subreddit,score,ups,downs,date,created_utc,parent_comment,clean_comment
0,1,"Good thing Soros is so rich, these guys must b...",zaures,politics,7,-1,-1,2016-11,2016-11-16 02:12:37,Anti-Trump protests not letting up for sixth s...,good thing soros is so rich these guys must be...
1,0,I'm also not being able to send NZB files from...,pienocake,usenet,1,1,0,2016-03,2016-03-17 03:23:56,Congratulations! SAB has always been a great p...,im also not being able to send nzb files from ...
2,1,Or Kanye West?,griffin852,Monstercat,2,2,0,2014-08,2014-08-13 00:10:39,Or Concept?,or kanye west
3,0,Fuck.,EL_SUPER_BEASTO,Flyers,3,3,0,2014-01,2014-01-15 23:55:51,Hartnell listed as day to day after blocking shot,fuck
4,0,"I think the suit does have some tech in it, si...",zoahporre,movies,2,2,0,2016-02,2016-02-11 18:44:38,"Ok, so either Batman's *normal* suit has some ...",i think the suit does have some tech in it sim...


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [6]:
tokens = tokenizer(df['comment'].iloc[0])
print(tokens)

{'input_ids': [101, 2204, 2518, 2061, 7352, 2003, 2061, 4138, 1010, 2122, 4364, 2442, 2022, 2437, 1037, 4288, 2006, 12253, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
tokenizer.convert_ids_to_tokens(tokens['input_ids'])

['[CLS]',
 'good',
 'thing',
 'so',
 '##ros',
 'is',
 'so',
 'rich',
 ',',
 'these',
 'guys',
 'must',
 'be',
 'making',
 'a',
 'killing',
 'on',
 'overtime',
 '.',
 '[SEP]']

In [8]:
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

In [9]:
df = df[df['clean_comment'].notna()]

In [10]:
print(type(df['clean_comment'].iloc[0]))
print(df['clean_comment'].iloc[0])

<class 'str'>
good thing soros is so rich these guys must be making a killing on overtime


In [11]:
df_bert = df[['clean_comment', 'label']].copy()

df_bert = df_bert.rename(
    columns={'clean_comment': 'text',
             'label' : 'labels'}
    )

dataset = Dataset.from_pandas(df_bert)

dataset = dataset.map(tokenize, batched=True)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/99777 [00:00<?, ? examples/s]

In [12]:
dataset[0]

{'text': 'good thing soros is so rich these guys must be making a killing on overtime',
 'labels': 1,
 '__index_level_0__': 0,
 'input_ids': [101,
  2204,
  2518,
  2061,
  7352,
  2003,
  2061,
  4138,
  2122,
  4364,
  2442,
  2022,
  2437,
  1037,
  4288,
  2006,
  12253,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [13]:
train_df, test_df = train_test_split(
    df_bert,
    test_size=0.2,
    random_state=14
)

In [14]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

Map:   0%|          | 0/79821 [00:00<?, ? examples/s]

Map:   0%|          | 0/19956 [00:00<?, ? examples/s]

In [15]:
train_dataset.features

{'text': Value(dtype='string', id=None),
 'labels': Value(dtype='int64', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [16]:
train_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

test_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)